# Parametrización

<div style="text-align: justify;">

Dentro de este _notebook_ se desarrolla el paso de parametrización de las variantes de las metaheurísticas construidas durante este trabajo de titulación. Particularmente, se requiere obtener un conjunto de hiperparámetros para cada versión del _NSGA-II_ que se evaluará, con un enfoque que busca ser **lo más justo posible entre las variantes**, de modo de asegurar la mejor configuración para cada una y que su evaluación posterior sea consistente.

Bajo este enfoque, se reconocen los siguientes parámetros dentro de la implementación actual (Tabla X):

| Parámetro                | Descripción                                                            | Variante     | Calibración  | Rango |
|--------------------------|------------------------------------------------------------------------|--------------|--------------|-----------------------------------------------|
| `ge_matrix`              | Matriz de distancias de expresión ($1 - S_{DE}$)                       | Todas        | No requerida | -                                             |
| `bi_matrix`              | Matriz de distancias biológicas ($1 - S_{DB}$)                         | Todas        | No requerida | -                                             |
| `n`                      | Número de genes (orden de las matrices)                                | Todas        | No requerida | [~2.000, ~11.000] genes                       |
| `k`                      | Número de medoides (_clusters_) por solución                           | Todas        | No requerida | 4                                             |
| `pop_size`               | Tamaño de la población                                                 | Todas        | No requerida | 150 individuos                                |
| `max_obj_function_calls` | Presupuesto global de evaluaciones, que es también el criterio de paro | Todas        | No requerida | 250.000 llamadas                              |
| `crossover_prob`         | Probabilidad de _crossover_ entre miembros de la población             | Todas        | Requerida    | [0,80; 0,90] (paso de 0,01)                   |
| `mutation_prob`          | Probabilidad de que un hijo presente mutación                          | Todas        | Requerida    | [0,01; 0,05] (paso de 0,01)                   |
| `seed`                   | Semilla del generador pseudoaleatorio                                  | Todas        | No requerida | 42                                            |
| `pls_local_budget`       | Evaluaciones que PLS puede usar por generación                         | `nsga2_PLS`  | Requerida    | [5, 10] % del total                           |
| `mols_local_budget`      | Evaluaciones que MOLS puede usar por generación                        | `nsga2_MOLS` | Requerida    | [5, 10] % del total                           |
| `bcs_local_budget`       | Evaluaciones que BCS puede usar por generación                         | `nsga2_BCS`  | Requerida    | [5, 10] % del total                           |
| `mbs_local_budget`       | Evaluaciones que MBS puede usar por generación                         | `nsga2_MBS`  | Requerida    | [5, 10] % del total                           |
| `neighborhood`           | Radio sobre $M_V = \sqrt{GE^2 + BI^2}$                                 | `nsga2_MOLS` | Requerida    | Percentil [5, 10] de la distribución de $M_V$ |
| `p`                      | Percentil de vecindad de cada ancla                                    | `nsga2_BCS`  | Requerida    | [5, 10] % de la vecindad                      |
| `mode`                   | Tipo de matriz de inconsistencia que usa el algoritmo                  | `nsga2_MBS`  | Requerida    | `"bio"`, `"no_discrepancy"`, `"expresion"`    |

Cabe considerar que todos aquellos parámetros que requieren calibración serán parametrizados según el mejor rendimiento, medido como la mejora del hipervolumen final del frente $F_1$. Este es, además, el criterio para seleccionar los parámetros definitivos en la etapa de ejecución de los experimentos.

</div>

In [1]:
######################################################
# MODULO DE PREPARACIÓN DEL AMBIENTE
# Detección de entorno (Colab / Local), rutas del repositorio, instalación
# de Optuna y configuración común del optimizador para todas las variantes.
#######################################################

############################
# Importaciones
############################
import sys                                                  # Ajuste de sys.path para importar los módulos de Source Code.
import os                                                   # Llamadas al SO.
import subprocess                                           # Instalación de dependencias faltantes.
import numpy  as np                                         # Aplicación de operaciones matematicas eficientes.
from pathlib import Path                                    # Aplicar un formato generalizado de direcciones en el script.

############################
# Detección de entorno (Colab / Local)
############################
# En Colab (incluido VSCode conectado a un runtime de Colab) el repositorio se
# encuentra clonado en /content/Thesis. En local, se busca la raíz del
# repositorio subiendo desde el directorio de trabajo actual.
try:
    import google.colab                                     # Módulo disponible solo dentro de runtimes de Colab.
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    THESIS_DIR = Path("/content/Thesis")
else:
    _cwd = Path(os.getcwd()).resolve()
    THESIS_DIR = next(
        (p for p in (_cwd, *_cwd.parents) if (p / "Datasets").is_dir() and (p / "Source Code").is_dir()),
        None,
    )
    if THESIS_DIR is None:
        raise FileNotFoundError(f"No se encontró la raíz del repositorio (Datasets/ y Source Code/) desde {_cwd}")

# Administración de directorios de matrices y código fuente.
MATRICES_DIR = THESIS_DIR / "Datasets" / "Matrices"
print(f"[entorno] {'Colab' if IN_COLAB else 'Local'} → {THESIS_DIR}")

SOURCE_CODE_DIR = THESIS_DIR / "Source Code"
if str(SOURCE_CODE_DIR) not in sys.path:
    sys.path.append(str(SOURCE_CODE_DIR))

############################
# Optuna (instalación si no está disponible)
############################
# plotly se requiere solo para optuna.visualization (historial de optimización,
# importancia de parámetros, etc.).
try:
    import optuna
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna", "plotly"])
    import optuna

from optuna.samplers import TPESampler                      # Muestreador bayesiano (Tree-structured Parzen Estimator).
optuna.logging.set_verbosity(optuna.logging.WARNING)        # Se silencia el log por trial; el progreso se reporta desde el propio notebook.
print(f"[optuna] versión {optuna.__version__}")

############################
# Configuración común del optimizador
############################
SEED = 42                                                   # Semilla única: misma para sampler y corridas (parámetro `seed` de la tabla).

# Almacenamiento persistente (SQLite): permite reanudar un estudio si el
# runtime se corta. En Colab el disco es efímero, por lo que conviene que
# RESULTS_DIR apunte a Google Drive si se quiere conservar entre sesiones.
RESULTS_DIR = THESIS_DIR / "Results" / "Parametrizacion"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def storage_url(nombre_variante: str) -> str:
    """Un archivo .db por variante: estudios independientes y fáciles de respaldar."""
    return f"sqlite:///{RESULTS_DIR / (nombre_variante + '.db')}"

def crear_sampler() -> TPESampler:
    """
    Mismo muestreador y misma semilla para TODAS las variantes, de modo que
    la diferencia entre estudios provenga del algoritmo y no del optimizador.
    """
    return TPESampler(seed=SEED, multivariate=True)

print(f"[resultados] {RESULTS_DIR}")

[entorno] Local → C:\Users\benja\Desktop\workspace\Thesis
[optuna] versión 5.0.0
[resultados] C:\Users\benja\Desktop\workspace\Thesis\Results\Parametrizacion


In [2]:
#######################################################
# MODULO DE IMPORTACIÓN DEL CÓDIGO FUENTE
# Se reutilizan los módulos validados en las iteraciones anteriores
# (Notes_1, Notes_2 y Notes_3) sin duplicar su lógica.
# Requiere que SOURCE_CODE_DIR ya esté en sys.path (módulo de ambiente).
#######################################################

############################
# Importaciones propias
############################
import solution_encoding                                    # Se importa el módulo completo: OBJ_FUNCTION_CALLS es un contador global
                                                            # que debe leerse/reiniciarse como solution_encoding.OBJ_FUNCTION_CALLS.
from solution_encoding import load_distance_matrix, random_medoids_pop
from pareto_sorting import non_dominated_sort               # Frentes de Pareto (para obtener F1).
from performance    import hypervolume_from_origin          # Métrica de desempeño: XB_GE · XB_BI (menor es mejor).

# Variante base.
from nsga2 import run_nsga2
# Variantes con operadores de búsqueda local clásicos (Notes_2).
from nsga2_variants import (
    run_nsga2_MOPR,                                         # + Multi-Objective Path-Relinking.
    run_nsga2_PLS,                                          # + Pareto Local Search.
    run_nsga2_MOLS,                                         # + L-MOLS / N-MOLS (según `mode`).
)

# Variantes con operadores de información biológica (Notes_3).
from nsga2_variants_bio import (
    run_nsga2_BCS,                                          # + Bio-Counterweight Search.
    run_nsga2_MBS,                                          # + Medoid-Based Search (según `mode`).
    MBS_MODES,                                              # ("bio", "no_discrepancy", "expresion").
)

# Matriz de vecindario M_V = sqrt(GE² + BI²): necesaria para fijar `neighborhood` en MOLS.
from LS import build_neighborhood_matrix

############################
# Verificación de importaciones
############################
VARIANTES = {
    "NSGA2":       run_nsga2,
    "NSGA2_MOPR":  run_nsga2_MOPR,
    "NSGA2_PLS":   run_nsga2_PLS,
    "NSGA2_MOLS":  run_nsga2_MOLS,
    "NSGA2_BCS":   run_nsga2_BCS,
    "NSGA2_MBS":   run_nsga2_MBS,
}
print(f"[importación] {len(VARIANTES)} funciones de variantes disponibles: {', '.join(VARIANTES)}")
print(f"[importación] Modos MBS: {MBS_MODES}")

[backend] CuPy detectado → operaciones vectorizadas en GPU.
[backend] CuPy detectado → operaciones vectorizadas en GPU.
[importación] 6 funciones de variantes disponibles: NSGA2, NSGA2_MOPR, NSGA2_PLS, NSGA2_MOLS, NSGA2_BCS, NSGA2_MBS
[importación] Modos MBS: ('bio', 'no_discrepancy', 'expresion')


## Optuna

Optuna es un _framework_ de código abierto enfocado en producir  

In [4]:
#######################################################
# MODULO DE PARAMETRIZACIÓN CON OPTUNA
# Un estudio por variante. En cada ejecución (trial) se evalúa la MISMA
# configuración sobre los datasets A1 y A2 (250.000 llamadas a la función
# objetivo por dataset, sin compartir presupuesto) y se promedia la mejora
# relativa del hipervolumen del frente F1.
#######################################################

############################
# Importaciones
############################
import io                                                   # Silenciar el log por generación de los algoritmos.
import json                                                 # Guardado de la mejor configuración.
import time                                                 # Tiempo por ejecución.
import contextlib                                           # redirect_stdout.
import optuna.visualization as vis                          # plot_optimization_history, plot_param_importances, plot_rank.

############################
# Configuración del usuario
############################
# Número TOTAL de ejecuciones (trials) por metaheurística. Cada una recorre
# ambos datasets. Si el estudio ya existe en disco, solo se completan las que faltan.
N_TRIALS = {
    "NSGA2":        10,
    "NSGA2_PLS":    10,
    "NSGA2_LMOLS":  10,
    "NSGA2_NMOLS":  10,
    "NSGA2_BCS":    10,
    "NSGA2_MBS":    10,
    "NSGA2_MOPR":   10,
}

############################
# Constantes del experimento (según la tabla de parametrización)
############################
DATASETS               = ["A1", "A2"]                       # Datasets de parametrización (prefijo A).
K                      = 4                                  # Medoides por solución.
POP_SIZE               = 150                                # Tamaño de la población.
MAX_OBJ_FUNCTION_CALLS = 50_000                            # Presupuesto POR ejecución y POR dataset.
PERCENTILES_NEIGHBORHOOD = range(5, 11)                     # Percentil de M_V para `neighborhood`: [5, 10].

FUNCIONES_OPTUNA = {
    "NSGA2":        run_nsga2,
    "NSGA2_PLS":    run_nsga2_PLS,
    "NSGA2_LMOLS":  run_nsga2_MOLS,
    "NSGA2_NMOLS":  run_nsga2_MOLS,
    "NSGA2_BCS":    run_nsga2_BCS,
    "NSGA2_MBS":    run_nsga2_MBS,
    "NSGA2_MOPR":   run_nsga2_MOPR,
}
assert set(N_TRIALS) == set(FUNCIONES_OPTUNA), "N_TRIALS debe definir todas las variantes."

############################
# Carga de datasets (una sola vez)
############################
def cargar_dataset(nombre: str) -> dict:
    """Carga GE/BI y precalcula los radios `neighborhood` (percentiles de M_V, sin diagonal)."""
    ge, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DE.npz")
    bi, _ = load_distance_matrix(MATRICES_DIR / f"{nombre}_S_DB_M.npz")
    n = ge.shape[0]

    m_v = build_neighborhood_matrix(ge, bi)
    valores = m_v[~np.eye(n, dtype=bool)]                   # M_V es simétrica: los duplicados no alteran el percentil.
    radios = dict(zip(
        PERCENTILES_NEIGHBORHOOD,
        np.percentile(valores, list(PERCENTILES_NEIGHBORHOOD)).astype(float),
    ))
    del m_v, valores

    return {"ge": ge, "bi": bi, "n": n, "neighborhood": radios}

DATOS = {nombre: cargar_dataset(nombre) for nombre in DATASETS}

############################
# Función objetivo
############################
def _sugerir_configuracion(trial: optuna.Trial, variante: str) -> dict:
    """Muestrea los hiperparámetros de la variante (rangos de la tabla). Es UNA configuración para ambos datasets."""
    cfg = {
        "crossover_prob": round(trial.suggest_float("crossover_prob", 0.80, 0.90, step=0.01), 2),
        "mutation_prob":  round(trial.suggest_float("mutation_prob",  0.01, 0.05, step=0.01), 2),
    }
    if variante == "NSGA2_PLS":
        cfg["pls_local_budget_pct"] = trial.suggest_int("pls_local_budget_pct", 5, 10)
    elif variante in ("NSGA2_LMOLS", "NSGA2_NMOLS"):
        cfg["mols_local_budget_pct"] = trial.suggest_int("mols_local_budget_pct", 5, 10)
        cfg["neighborhood_pct"]      = trial.suggest_int("neighborhood_pct", 5, 10)
    elif variante == "NSGA2_BCS":
        cfg["bcs_local_budget_pct"] = trial.suggest_int("bcs_local_budget_pct", 5, 10)
        cfg["p_pct"]                = trial.suggest_int("p_pct", 5, 10)
    elif variante == "NSGA2_MBS":
        cfg["mbs_local_budget_pct"] = trial.suggest_int("mbs_local_budget_pct", 5, 10)
        cfg["mode"]                 = trial.suggest_categorical("mode", list(MBS_MODES))
    return cfg

def _kwargs_variante(variante: str, cfg: dict, ds: dict) -> dict:
    """Traduce la configuración muestreada a los argumentos reales de la función de la variante."""
    kw = dict(
        ge_matrix=ds["ge"], bi_matrix=ds["bi"], n=ds["n"], k=K,
        pop_size=POP_SIZE, max_obj_function_calls=MAX_OBJ_FUNCTION_CALLS,
        crossover_prob=cfg["crossover_prob"], mutation_prob=cfg["mutation_prob"],
        seed=SEED,
    )
    budget = lambda pct: MAX_OBJ_FUNCTION_CALLS * pct // 100

    if variante == "NSGA2_PLS":
        kw["pls_local_budget"] = budget(cfg["pls_local_budget_pct"])
    elif variante in ("NSGA2_LMOLS", "NSGA2_NMOLS"):
        kw["mode"]              = "l_mols" if variante == "NSGA2_LMOLS" else "n_mols"
        kw["neighborhood"]      = ds["neighborhood"][cfg["neighborhood_pct"]]
        kw["mols_local_budget"] = budget(cfg["mols_local_budget_pct"])
    elif variante == "NSGA2_BCS":
        kw["p"]                = cfg["p_pct"] / 100
        kw["bcs_local_budget"] = budget(cfg["bcs_local_budget_pct"])
    elif variante == "NSGA2_MBS":
        kw["mode"]             = cfg["mode"]
        kw["mbs_local_budget"] = budget(cfg["mbs_local_budget_pct"])
    return kw

def objective(trial: optuna.Trial, variante: str) -> float:
    """
    Retorna la MEJORA RELATIVA promedio (entre datasets) del hipervolumen medio de F1:
        mejora = (HV_F1_inicial − HV_F1_final) / HV_F1_inicial     (se MAXIMIZA)
    El hipervolumen del proyecto es XB_GE·XB_BI (menor es mejor), por lo que una
    mejora positiva indica que el algoritmo redujo el valor respecto a la población
    inicial. Al ser relativa, es comparable entre A1 y A2 aunque sus escalas difieran.
    """
    cfg = _sugerir_configuracion(trial, variante)
    mejoras = []

    for nombre, ds in DATOS.items():
        solution_encoding.OBJ_FUNCTION_CALLS = 0            # Reinicio obligatorio: el presupuesto NO se comparte entre datasets.
        t0 = time.time()
        with contextlib.redirect_stdout(io.StringIO()):     # Se silencia el print por generación.
            resultado = FUNCIONES_OPTUNA[variante](**_kwargs_variante(variante, cfg, ds))

        hv = resultado["history_hv_f1_mean"]
        mejora = (hv[0] - hv[-1]) / hv[0]
        mejoras.append(mejora)

        trial.set_user_attr(f"{nombre}_mejora_hv_f1", float(mejora))
        trial.set_user_attr(f"{nombre}_hv_f1_final", float(hv[-1]))
        trial.set_user_attr(f"{nombre}_generaciones", int(resultado["generations_run"]))
        trial.set_user_attr(f"{nombre}_llamadas", int(solution_encoding.OBJ_FUNCTION_CALLS))
        trial.set_user_attr(f"{nombre}_segundos", round(time.time() - t0, 1))

    return float(np.mean(mejoras))

############################
# Visualización por metaheurística
############################
def mostrar_graficos(estudio: optuna.Study) -> None:
    graficos = {
        "optimization_history": vis.plot_optimization_history,
        "param_importances":    vis.plot_param_importances,
        "rank":                 vis.plot_rank,
    }
    for nombre, funcion in graficos.items():
        try:
            fig = funcion(estudio)
            fig.update_layout(title=f"{estudio.study_name} — {nombre}")
            fig.write_html(RESULTS_DIR / f"{estudio.study_name}_{nombre}.html")
            fig.show()
        except Exception as e:                              # Ej.: muy pocos trials o parámetros sin variación.
            print(f"[{estudio.study_name}] no se pudo generar '{nombre}': {e}")

############################
# Ejecución
############################
def _log_trial(estudio: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    segundos = sum(v for k, v in trial.user_attrs.items() if k.endswith("_segundos"))
    print(f"  [{estudio.study_name}] trial {trial.number:>3} → mejora = {trial.value:+.4f}  "
          f"({segundos / 60:.1f} min)  {trial.params}")

MEJORES = {}
for variante in FUNCIONES_OPTUNA:
    estudio = optuna.create_study(
        study_name=variante,
        storage=storage_url(variante),
        sampler=crear_sampler(),
        direction="maximize",                               # Se maximiza la mejora del hipervolumen en F1.
        load_if_exists=True,                                # Permite reanudar un estudio interrumpido.
    )

    completados = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
    pendientes  = max(0, N_TRIALS[variante] - completados)
    print(f"\n════════ {variante}: {completados} completados, {pendientes} pendientes (total {N_TRIALS[variante]}) ════════")

    if pendientes:
        estudio.optimize(lambda t, v=variante: objective(t, v), n_trials=pendientes, callbacks=[_log_trial])

    MEJORES[variante] = {"mejora_hv_f1": estudio.best_value, "parametros": estudio.best_params}
    print(f"[{variante}] mejor mejora = {estudio.best_value:+.4f} | {estudio.best_params}")

    mostrar_graficos(estudio)

with open(RESULTS_DIR / "mejores_parametros.json", "w", encoding="utf-8") as f:
    json.dump(MEJORES, f, indent=2, ensure_ascii=False)
print(f"\n[guardado] {RESULTS_DIR / 'mejores_parametros.json'}")

[load] 'A1_S_DE.npz' → matriz 8018×8018. Primer gen: 'A1BG', último: 'ENSG00000285994'.
[load] 'A1_S_DB_M.npz' → matriz 8018×8018. Primer gen: 'A1BG', último: 'ENSG00000285994'.
[load] 'A2_S_DE.npz' → matriz 2705×2705. Primer gen: 'AKT3', último: 'ENSG00000285928'.
[load] 'A2_S_DB_M.npz' → matriz 2705×2705. Primer gen: 'AKT3', último: 'ENSG00000285928'.

════════ NSGA2: 10 completados, 0 pendientes (total 10) ════════
[NSGA2] mejor mejora = +0.6885 | {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.04}



════════ NSGA2_PLS: 0 completados, 10 pendientes (total 10) ════════
  [NSGA2_PLS] trial   0 → mejora = +0.4875  (0.2 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05, 'pls_local_budget_pct': 9}


C:\Users\benja\Desktop\workspace\Thesis\Source Code\pareto_sorting.py:97: RuntimeWarning:

invalid value encountered in scalar divide



  [NSGA2_PLS] trial   1 → mejora = -inf  (0.2 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.01, 'pls_local_budget_pct': 5}
  [NSGA2_PLS] trial   2 → mejora = +0.6551  (0.2 min)  {'crossover_prob': 0.8, 'mutation_prob': 0.05, 'pls_local_budget_pct': 8}
  [NSGA2_PLS] trial   3 → mejora = +0.5704  (0.2 min)  {'crossover_prob': 0.8700000000000001, 'mutation_prob': 0.01, 'pls_local_budget_pct': 10}
  [NSGA2_PLS] trial   4 → mejora = +0.7171  (0.2 min)  {'crossover_prob': 0.89, 'mutation_prob': 0.02, 'pls_local_budget_pct': 6}
  [NSGA2_PLS] trial   5 → mejora = +0.6890  (0.2 min)  {'crossover_prob': 0.8200000000000001, 'mutation_prob': 0.02, 'pls_local_budget_pct': 8}
  [NSGA2_PLS] trial   6 → mejora = +0.7092  (0.2 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.02, 'pls_local_budget_pct': 8}
  [NSGA2_PLS] trial   7 → mejora = +0.7134  (0.2 min)  {'crossover_prob': 0.81, 'mutation_prob': 0.02, 'pls_local_budget_pct': 7}
  [NSGA2_PLS] trial   8 → mejora = +0

[W 2026-10-07 15:32:01,593] Trial 1 is omitted in visualization because its objective value is inf or nan.


...


════════ NSGA2_NMOLS: 0 completados, 10 pendientes (total 10) ════════
  [NSGA2_NMOLS] trial   0 → mejora = +0.5084  (0.2 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05, 'mols_local_budget_pct': 9, 'neighborhood_pct': 8}
  [NSGA2_NMOLS] trial   1 → mejora = +0.5845  (0.2 min)  {'crossover_prob': 0.81, 'mutation_prob': 0.01, 'mols_local_budget_pct': 5, 'neighborhood_pct': 10}
  [NSGA2_NMOLS] trial   2 → mejora = +0.5303  (0.2 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.04, 'mols_local_budget_pct': 5, 'neighborhood_pct': 10}
  [NSGA2_NMOLS] trial   3 → mejora = +0.4629  (0.3 min)  {'crossover_prob': 0.89, 'mutation_prob': 0.02, 'mols_local_budget_pct': 6, 'neighborhood_pct': 6}
  [NSGA2_NMOLS] trial   4 → mejora = +0.6107  (0.2 min)  {'crossover_prob': 0.8300000000000001, 'mutation_prob': 0.03, 'mols_local_budget_pct': 7, 'neighborhood_pct': 6}
  [NSGA2_NMOLS] trial   5 → mejora = +0.3287  (0.2 min)  {'crossover_prob': 0.8600000000000001, 'mutatio


════════ NSGA2_BCS: 0 completados, 10 pendientes (total 10) ════════
  [NSGA2_BCS] trial   0 → mejora = +0.6633  (1.1 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05, 'bcs_local_budget_pct': 9, 'p_pct': 8}
  [NSGA2_BCS] trial   1 → mejora = +0.6150  (0.8 min)  {'crossover_prob': 0.81, 'mutation_prob': 0.01, 'bcs_local_budget_pct': 5, 'p_pct': 10}
  [NSGA2_BCS] trial   2 → mejora = +0.6269  (0.9 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.04, 'bcs_local_budget_pct': 5, 'p_pct': 10}
  [NSGA2_BCS] trial   3 → mejora = +0.6356  (1.5 min)  {'crossover_prob': 0.89, 'mutation_prob': 0.02, 'bcs_local_budget_pct': 6, 'p_pct': 6}
  [NSGA2_BCS] trial   4 → mejora = +0.4605  (1.4 min)  {'crossover_prob': 0.8300000000000001, 'mutation_prob': 0.03, 'bcs_local_budget_pct': 7, 'p_pct': 6}
  [NSGA2_BCS] trial   5 → mejora = +0.5780  (1.2 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.01, 'bcs_local_budget_pct': 6, 'p_pct': 7}
  [NSGA2_BCS] trial 


════════ NSGA2_MBS: 0 completados, 10 pendientes (total 10) ════════
  [NSGA2_MBS] trial   0 → mejora = +0.6358  (0.2 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05, 'mbs_local_budget_pct': 9, 'mode': 'bio'}
  [NSGA2_MBS] trial   1 → mejora = +0.7410  (0.2 min)  {'crossover_prob': 0.8, 'mutation_prob': 0.05, 'mbs_local_budget_pct': 8, 'mode': 'expresion'}
  [NSGA2_MBS] trial   2 → mejora = +0.7683  (0.2 min)  {'crossover_prob': 0.89, 'mutation_prob': 0.02, 'mbs_local_budget_pct': 6, 'mode': 'expresion'}
  [NSGA2_MBS] trial   3 → mejora = +0.7097  (0.2 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.02, 'mbs_local_budget_pct': 8, 'mode': 'expresion'}
  [NSGA2_MBS] trial   4 → mejora = +0.6061  (0.2 min)  {'crossover_prob': 0.8500000000000001, 'mutation_prob': 0.04, 'mbs_local_budget_pct': 6, 'mode': 'no_discrepancy'}
  [NSGA2_MBS] trial   5 → mejora = +0.7096  (0.2 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.01, 'mbs_local_budget_


════════ NSGA2_MOPR: 0 completados, 10 pendientes (total 10) ════════
  [NSGA2_MOPR] trial   1 → mejora = +0.6339  (12.8 min)  {'crossover_prob': 0.8400000000000001, 'mutation_prob': 0.05}
  [NSGA2_MOPR] trial   2 → mejora = +0.6828  (15.0 min)  {'crossover_prob': 0.88, 'mutation_prob': 0.03}
  [NSGA2_MOPR] trial   3 → mejora = +0.6991  (13.0 min)  {'crossover_prob': 0.81, 'mutation_prob': 0.01}
  [NSGA2_MOPR] trial   4 → mejora = +0.6736  (15.4 min)  {'crossover_prob': 0.8, 'mutation_prob': 0.05}
  [NSGA2_MOPR] trial   5 → mejora = +0.6948  (15.9 min)  {'crossover_prob': 0.8600000000000001, 'mutation_prob': 0.04}
  [NSGA2_MOPR] trial   6 → mejora = +0.6736  (15.3 min)  {'crossover_prob': 0.8, 'mutation_prob': 0.05}
  [NSGA2_MOPR] trial   7 → mejora = +0.6627  (13.9 min)  {'crossover_prob': 0.89, 'mutation_prob': 0.02}
  [NSGA2_MOPR] trial   8 → mejora = +0.7069  (14.3 min)  {'crossover_prob': 0.8200000000000001, 'mutation_prob': 0.01}
  [NSGA2_MOPR] trial   9 → mejora = +0.6783  (13.


[guardado] C:\Users\benja\Desktop\workspace\Thesis\Results\Parametrizacion\mejores_parametros.json
